# LP-Prop v2 · Notebook 2: truy xuất (s3), reader và judge (s5), đánh giá (s6)

**Input:**
- Output của **Notebook 1** (*Add Input → Your Work → Notebooks*).
- `lp-prop-pipeline-s3`: chỉ để dùng lại embedding cũ, không đọc dữ liệu cũ nào khác.

**Accelerator:** GPU T4 ×2. **Internet:** On.

| Bước | Việc | Thời gian ước tính (chưa đo) |
|---|---|---|
| s3 | dùng lại embedding cũ, embed câu hỏi mới, truy xuất 8 hệ | 20–40 phút |
| s5 reader | Qwen2.5-7B-AWQ, 5 hệ × 2 budget + oracle = 13.200 câu trả lời | 1,5–2,5 giờ |
| s5 judge | Qwen2.5-14B-AWQ (2 GPU): kiểm tra 100 câu rồi chấm cả bộ | 3,5–4,5 giờ |
| s6 | Tầng 1–2 + bootstrap | vài phút |

Tổng **khoảng 6–7,5 giờ**, nằm trong giới hạn 12 giờ của Kaggle. Chạy bằng **Save Version → Save & Run All**. Kết quả được nén thành `lp_prop_v2_results.zip`.

In [ ]:
!pip install -q vllm
# vLLM nâng torch lên CUDA 13; torchaudio cũ làm lỗi import -> gỡ (không cần)
!pip uninstall -y -q torchaudio
!nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv
!pip install -q sentence-transformers faiss-cpu

In [ ]:
%%writefile /kaggle/working/s3_retrieve_v2.py
"""
s3_retrieve_v2.py — LP-Prop v2: truy xuất BGE-M3 cho 8 hệ, DÙNG LẠI embedding cũ của s3.

- Đơn vị (passage / proposition) của corpus v2 là tập con của corpus cũ -> lấy vector từ
  emb_<set>_bge-m3.npy cũ theo emb_map.json (do s1_clean tạo). Chỉ embed: passage mới + câu hỏi.
- Nếu không tìm thấy file cũ hoặc số dòng không khớp, tự embed lại toàn bộ tập đó (chậm hơn, vẫn đúng).
- Logic fill_budget / ranked_aids / con trỏ giống hệt run_s3_s6.py cũ.

Ra (trong --out): ranks_<hệ>.jsonl (top-20 Điều cho mọi câu hỏi eval), contexts_<hệ>_B<budget>.jsonl (câu cond), s3_meta.json

Dùng:
  python s3_retrieve_v2.py --data /kaggle/working/data_v2 --cond <file 1200 câu> --old_emb_root /kaggle/input --out /kaggle/working/work_s3v2
"""
import argparse, glob, json, os, time
from collections import OrderedDict, defaultdict
import numpy as np

SYSTEMS = {"passage": ("passage", "none"), "std": ("std", "none"), "lp_noptr": ("lp", "none"),
           "lp_out": ("lp", "out"), "lp": ("lp", "both"), "std_raw": ("std_raw", "none"),
           "lp_raw": ("lp_raw", "both"), "lp_edge_only": ("lp_edge_only", "both")}
ORDER = ["passage", "std_raw", "std", "lp_noptr", "lp_out", "lp_raw", "lp_edge_only", "lp"]
UNIT_SETS = ["passage", "std", "lp", "std_raw", "lp_raw", "lp_edge_only"]


def read_jsonl(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for l in f:
            l = l.strip()
            if l:
                try:
                    out.append(json.loads(l))
                except json.JSONDecodeError:
                    pass
    return out

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def build_pointers(records):
    props, condition_ids, rule_of = OrderedDict(), defaultdict(list), defaultdict(list)
    for r in records:
        for p in r["props"]:
            props[p["uid"]] = p
        for e in r.get("pointers", []):
            if e["tail"] not in condition_ids[e["head"]]:
                condition_ids[e["head"]].append(e["tail"])
            if e["head"] not in rule_of[e["tail"]]:
                rule_of[e["tail"]].append(e["head"])
    return props, dict(condition_ids), dict(rule_of)

def fill_budget(ranked_uids, units, B, condition_ids=None, rule_of=None, mode="none"):
    condition_ids, rule_of = condition_ids or {}, rule_of or {}
    ctx, seen, used = [], set(), 0
    for uid in ranked_uids:
        if used >= B:
            break
        group = [uid]
        if mode in ("out", "both"):
            group = group + condition_ids.get(uid, [])
        if mode == "both":
            group = rule_of.get(uid, []) + group
        for q in group:
            if q in seen or used >= B or q not in units:
                continue
            w = units[q]["text"].split()[: B - used]
            ctx.append({"uid": q, "aid": units[q]["aid"], "text": " ".join(w)})
            seen.add(q); used += len(w)
    return ctx

def ranked_aids(ranked_uids, units):
    seen, out = set(), []
    for u in ranked_uids:
        a = units[u]["aid"]
        if a not in seen:
            seen.add(a); out.append(a)
    return out


class Encoder:
    def __init__(self, name, mock=False, dim=1024):
        self.mock, self.dim = mock, dim
        if mock:
            return
        import torch
        from sentence_transformers import SentenceTransformer
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"[encoder] {name} trên {dev}", flush=True)
        self.m = SentenceTransformer(name, device=dev); self.m.max_seq_length = 512
        if dev == "cuda":
            self.m.half()

    def encode(self, texts, batch=128):
        if not texts:
            return np.zeros((0, self.dim), dtype="float32")
        if self.mock:
            rs = [np.random.default_rng(abs(hash(t)) % (2 ** 32)).standard_normal(self.dim) for t in texts]
            X = np.asarray(rs, dtype="float32"); return X / np.linalg.norm(X, axis=1, keepdims=True)
        return self.m.encode(texts, batch_size=batch, normalize_embeddings=True, show_progress_bar=True,
                             convert_to_numpy=True).astype("float32")


def search(X, Q, k):
    """Top-k inner product. Dùng faiss nếu có, không thì torch/numpy theo lô."""
    k = min(k, len(X))
    try:
        import faiss
        ix = faiss.IndexFlatIP(X.shape[1]); ix.add(X)
        return ix.search(Q, k)[1]
    except ImportError:
        pass
    try:
        import torch
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        Xt = torch.from_numpy(X).to(dev, dtype=torch.float16 if dev == "cuda" else torch.float32)
        out = []
        for i in range(0, len(Q), 2048):
            q = torch.from_numpy(Q[i:i + 2048]).to(dev, dtype=Xt.dtype)
            out.append(torch.topk(q @ Xt.T, k, dim=1).indices.cpu().numpy())
        return np.vstack(out)
    except ImportError:
        out = []
        for i in range(0, len(Q), 1024):
            s = Q[i:i + 1024] @ X.T
            idx = np.argpartition(-s, k - 1, axis=1)[:, :k]
            out.append(np.take_along_axis(idx, np.argsort(-np.take_along_axis(s, idx, 1), 1), 1))
        return np.vstack(out)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="/kaggle/working/data_v2")
    ap.add_argument("--cond", required=True, help="file câu hỏi cond (1.200 câu)")
    ap.add_argument("--old_emb_root", default="/kaggle/input")
    ap.add_argument("--out", default="/kaggle/working/work_s3v2")
    ap.add_argument("--encoder", default="BAAI/bge-m3")
    ap.add_argument("--budgets", type=int, nargs="+", default=[100, 500])
    ap.add_argument("--topk", type=int, default=100)
    ap.add_argument("--mock", action="store_true")
    ap.add_argument("--mock_dim", type=int, default=64)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    t0 = time.time()
    tag = a.encoder.split("/")[-1]
    emb_map = json.load(open(os.path.join(a.data, "emb_map.json")))

    unit_sets = {"passage": (OrderedDict((u["uid"], u) for u in read_jsonl(os.path.join(a.data, "passages.jsonl"))), {}, {})}
    for m in UNIT_SETS[1:]:
        unit_sets[m] = build_pointers(read_jsonl(os.path.join(a.data, f"props_{m}.jsonl")))

    arts = {x["aid"]: x for x in read_jsonl(os.path.join(a.data, "articles_v2.jsonl"))}
    cond = [dict(q, kind="cond") for q in read_jsonl(a.cond) if q["aid"] in arts]
    gen = [dict(q, kind="generic") for q in read_jsonl(os.path.join(a.data, "qa_generic.jsonl"))
           if arts.get(q["aid"], {}).get("split") == "eval"]
    qs = cond + gen
    assert len({q["qid"] for q in qs}) == len(qs), "qid trùng giữa câu cond và generic"
    cond_ids = {q["qid"] for q in cond}
    print(f"[data] {len(cond)} câu cond + {len(gen)} câu generic eval", flush=True)

    enc = Encoder(a.encoder, mock=a.mock, dim=a.mock_dim)
    qc = os.path.join(a.out, f"emb_queries_{tag}.npy")
    if os.path.exists(qc) and len(np.load(qc)) == len(qs):
        Q = np.load(qc)
    else:
        Q = enc.encode([q["question"] for q in qs]); np.save(qc, Q)

    meta = {"n_cond": len(cond), "n_generic": len(gen), "units": {}, "emb": {}}
    for name in UNIT_SETS:
        units, c_ids, r_of = unit_sets[name]
        uids = list(units)
        rows = emb_map[name]["rows"]
        assert len(rows) == len(uids), f"emb_map lệch với {name}"
        hits = sorted(glob.glob(os.path.join(a.old_emb_root, "**", f"emb_{name}_{tag}.npy"), recursive=True))
        X = None
        if hits:
            old = np.load(hits[0], mmap_mode="r")
            if len(old) == emb_map[name]["n_old"]:
                X = np.zeros((len(uids), old.shape[1]), dtype="float32")
                keep = [i for i, r in enumerate(rows) if r >= 0]
                X[keep] = old[[rows[i] for i in keep]]
                # kiểm tra an toàn: embed lại 32 đơn vị ngẫu nhiên, phải gần trùng vector cũ
                probe = np.random.default_rng(0).choice(keep, size=min(32, len(keep)), replace=False) if keep else []
                cos = float(np.mean(np.sum(enc.encode([units[uids[i]]["text"] for i in probe]) * X[probe], axis=1))) if len(probe) and not a.mock else 1.0
                if cos < 0.97:
                    meta["emb"][name] = f"vector cũ không khớp văn bản (cos={cos:.3f}) -> embed lại toàn bộ"
                    X = None
                else:
                    new = [i for i, r in enumerate(rows) if r < 0]
                    if new:
                        X[new] = enc.encode([units[uids[i]]["text"] for i in new])
                    meta["emb"][name] = f"dùng lại {len(keep)} vector cũ (kiểm tra cos={cos:.3f}), embed mới {len(new)}"
            else:
                meta["emb"][name] = f"file cũ có {len(old)} dòng != {emb_map[name]['n_old']} -> embed lại toàn bộ"
        else:
            meta["emb"][name] = "không thấy file cũ -> embed lại toàn bộ"
        if X is None:
            X = enc.encode([units[u]["text"] for u in uids])
        print(f"[emb] {name}: {meta['emb'][name]}", flush=True)
        meta["units"][name] = len(uids)

        I = search(np.ascontiguousarray(X, dtype="float32"), np.ascontiguousarray(Q, dtype="float32"), a.topk)
        ranked = [[uids[j] for j in row if j >= 0] for row in I]
        for s in ORDER:
            src, mode = SYSTEMS[s]
            if src != name:
                continue
            write_jsonl(os.path.join(a.out, f"ranks_{s}.jsonl"),
                        [{"qid": q["qid"], "aids": ranked_aids(r, units)[:20]} for q, r in zip(qs, ranked)])
            for B in a.budgets:
                write_jsonl(os.path.join(a.out, f"contexts_{s}_B{B}.jsonl"),
                            [{"qid": q["qid"], "kind": "cond", "units": fill_budget(r, units, B, c_ids, r_of, mode)}
                             for q, r in zip(qs, ranked) if q["qid"] in cond_ids])
            print(f"[OK] {s}", flush=True)
    meta["minutes"] = round((time.time() - t0) / 60, 1)
    json.dump(meta, open(os.path.join(a.out, "s3_meta.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(json.dumps(meta, ensure_ascii=False, indent=1)); print("[done]")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/s5_reader_judge.py
"""
s5_reader_judge.py — Tầng 3: Reader + LLM-judge cho LP-Prop (bộ 1.200 câu cond, split=eval).

Đầu vào (đọc từ --root, thường là /kaggle/input):
  - cond_questions_eval_1200.jsonl      (S2 / work_clean)
  - contexts_{system}_B{budget}.jsonl   (S3 / work_s3)
  - articles_all.jsonl                  (D1, chỉ dùng cho hệ "oracle")

3 giai đoạn (--stage):
  reader    : sinh câu trả lời từ ngữ cảnh của từng hệ × budget
  judge     : chấm từng câu trả lời so với đáp án chuẩn (JSON: correct / condition_kept / overgeneralized)
  aggregate : gộp các shard, tính chỉ số + paired bootstrap -> results/s5_results.json, s5_tables.md

Có checkpoint: chạy lại lệnh cũ sẽ tự bỏ qua mục đã xong. Chia 2 GPU bằng --shard 0/1 --num_shards 2.

Ví dụ trên Kaggle (2×T4):
  CUDA_VISIBLE_DEVICES=0 python s5_reader_judge.py --stage reader --shard 0 --num_shards 2 &
  CUDA_VISIBLE_DEVICES=1 python s5_reader_judge.py --stage reader --shard 1 --num_shards 2 &
  wait
  (tương tự cho --stage judge), rồi: python s5_reader_judge.py --stage aggregate
"""
import argparse, glob, json, os, random, re, time, unicodedata, zlib
from collections import defaultdict

DEFAULT_MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"   # cùng model với s2
DEFAULT_SYSTEMS = ["passage", "std", "lp_noptr", "lp_out", "lp"]
ABLATION_SYSTEMS = ["std_raw", "lp_raw", "lp_edge_only"]
ORDER = ["oracle", "passage", "std_raw", "std", "lp_noptr", "lp_out", "lp_raw", "lp_edge_only", "lp"]
STOP = set("của và các là được có cho theo với trong này đó những một người phải thì".split())
ABSTAIN_PAT = re.compile(r"không đủ thông tin|không có thông tin|ngữ cảnh không (đề cập|cung cấp)", re.I)


# ---------------- tiện ích ----------------
def norm(s):
    return unicodedata.normalize("NFC", s or "")

def toks(s):
    return [w for w in re.findall(r"\w+", norm(s).lower()) if w not in STOP]

def coverage(span, texts):
    st = toks(span)
    if not st:
        return 0.0
    bag = set(w for t in texts for w in toks(t))
    return sum(1 for w in st if w in bag) / len(st)

def phrase_hit(span, text, n=4, frac=0.5):
    """Điều kiện xuất hiện theo CỤM từ liền nhau (n-gram) trong câu trả lời -> chặt hơn so khớp túi từ."""
    c, a = toks(span), toks(text)
    if len(c) < n:
        return coverage(span, [text]) >= 0.9 and " ".join(c) in " ".join(a)
    ag = {tuple(a[i:i + n]) for i in range(len(a) - n + 1)}
    cg = [tuple(c[i:i + n]) for i in range(len(c) - n + 1)]
    return sum(g in ag for g in cg) / len(cg) >= frac

def read_jsonl(path):
    out = []
    if not path or not os.path.exists(path):
        return out
    with open(path, encoding="utf-8") as f:
        for l in f:
            l = l.strip()
            if l:
                try:
                    out.append(json.loads(l))
                except json.JSONDecodeError:
                    continue
    return out

def append_jsonl(path, rows):
    with open(path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def find(root, name, prefer=None):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
    if not hits:
        return None
    if prefer:
        pref = [h for h in hits if prefer in h]
        if pref:
            return pref[0]
    return hits[0]

def shard_of(key, n):
    return zlib.crc32(key.encode("utf-8")) % n

mean = lambda xs: sum(xs) / len(xs) if xs else float("nan")
pct = lambda x: "–" if x != x else f"{100 * x:.1f}"


# ---------------- prompt ----------------
READER_SYS = (
    "Bạn là trợ lý pháp lý tiếng Việt. Chỉ dựa vào các đoạn NGỮ CẢNH được cung cấp để trả lời, "
    "không dùng kiến thức bên ngoài. Trả lời ngắn gọn, tối đa khoảng 150 từ. "
    "Nếu quy định có điều kiện, ngoại lệ, giới hạn hoặc trường hợp không áp dụng thì phải nêu rõ. "
    "Nếu ngữ cảnh không đủ để trả lời, hãy nói: \"Không đủ thông tin trong ngữ cảnh.\""
)

def reader_messages(question, units):
    if units:
        ctx = "\n".join(f"[{i + 1}] {u['text']}" for i, u in enumerate(units))
    else:
        ctx = "(trống)"
    user = f"NGỮ CẢNH:\n{ctx}\n\nCÂU HỎI: {question}\n\nTRẢ LỜI:"
    return [{"role": "system", "content": READER_SYS}, {"role": "user", "content": user}]

JUDGE_SYS = (
    "You are a strict but fair grader for Vietnamese legal question answering. "
    "Read carefully, reason briefly, then output a JSON object on the last line."
)

def judge_messages(q, answer):
    """Lượt 1 (main): chỉ chấm correct + rule_stated."""
    user = (
        f"QUESTION (Vietnamese):\n{q['question']}\n\n"
        f"REFERENCE ANSWER:\n{q['answer']}\n\n"
        f"CANDIDATE ANSWER:\n{answer}\n\n"
        f"LEGAL RULE: {q['head']}\n\n"
        "Answer two yes/no checks about the CANDIDATE ANSWER:\n"
        "1. correct: Does it give the same core answer to the QUESTION as the REFERENCE ANSWER, without contradicting it? "
        "Missing minor details is fine. Saying there is not enough information = 0.\n"
        "2. rule_stated: Does it state or apply the LEGAL RULE above (paraphrase is fine)?\n\n"
        "Write at most 2 short lines of reasoning, then on the LAST line output only:\n"
        '{"correct": <0 or 1>, "rule_stated": <0 or 1>}'
    )
    return [{"role": "system", "content": JUDGE_SYS}, {"role": "user", "content": user}]

COND_SYS = (
    "Bạn là người kiểm tra văn bản pháp luật tiếng Việt, làm việc cẩn thận và khách quan. "
    "Nhiệm vụ: xác định CÂU TRẢ LỜI có nêu ĐIỀU KIỆN/NGOẠI LỆ đã cho hay không."
)

COND_FEWSHOT = """Quy tắc chấm:
- Chấm 1 nếu câu trả lời nêu điều kiện/ngoại lệ đó, nguyên văn HOẶC diễn đạt lại cùng ý (đổi từ, đổi thứ tự, rút gọn hợp lý đều được).
- Chấm 0 nếu câu trả lời không nhắc tới điều kiện đó, chỉ nêu quy tắc chung, nêu một điều kiện KHÁC, hoặc nói không đủ thông tin.
Cách làm: trước hết chép lại ĐÚNG đoạn trong câu trả lời thể hiện điều kiện (hoặc ghi KHÔNG CÓ), rồi kết luận.

Ví dụ 1
QUY TẮC: Người lao động được nghỉ hằng năm 12 ngày làm việc
ĐIỀU KIỆN: nếu làm việc đủ 12 tháng cho một người sử dụng lao động
CÂU TRẢ LỜI: Người lao động làm việc đủ 12 tháng cho một người sử dụng lao động thì được nghỉ 12 ngày phép năm.
TRÍCH: "làm việc đủ 12 tháng cho một người sử dụng lao động thì được nghỉ"
KẾT LUẬN: 1

Ví dụ 2
QUY TẮC: Tổ chức tín dụng phải niêm yết công khai lãi suất
ĐIỀU KIỆN: trừ trường hợp quy định tại khoản 3 Điều này
CÂU TRẢ LỜI: Tổ chức tín dụng phải công khai lãi suất, ngoại trừ các trường hợp nêu tại khoản 3.
TRÍCH: "ngoại trừ các trường hợp nêu tại khoản 3"
KẾT LUẬN: 1

Ví dụ 3
QUY TẮC: Người nộp thuế được gia hạn nộp thuế
ĐIỀU KIỆN: trong trường hợp bị thiệt hại vật chất do thiên tai, hỏa hoạn
CÂU TRẢ LỜI: Người nộp thuế được gia hạn nộp thuế theo quy định của Luật Quản lý thuế, thời gian gia hạn tối đa 2 năm.
TRÍCH: KHÔNG CÓ
KẾT LUẬN: 0

Ví dụ 4
QUY TẮC: Doanh nghiệp phải thông báo cho cơ quan đăng ký kinh doanh
ĐIỀU KIỆN: nếu thay đổi người đại diện theo pháp luật
CÂU TRẢ LỜI: Không đủ thông tin trong ngữ cảnh.
TRÍCH: KHÔNG CÓ
KẾT LUẬN: 0
"""

def cond_messages(q, answer):
    """Lượt 2 (cond) v4: few-shot, bắt trích dẫn trước rồi mới kết luận."""
    user = (
        COND_FEWSHOT +
        "\nBây giờ chấm trường hợp sau. Trả lời đúng 2 dòng TRÍCH và KẾT LUẬN như ví dụ.\n"
        f"QUY TẮC: {q['head']}\n"
        f"ĐIỀU KIỆN: {q['condition']}\n"
        f"CÂU TRẢ LỜI: {answer}\n"
    )
    return [{"role": "system", "content": COND_SYS}, {"role": "user", "content": user}]

def parse_main(raw):
    for m in reversed(list(re.finditer(r"\{[^{}]*\}", raw or ""))):
        d = {k: re.search(rf'"?{k}"?\s*:\s*"?([01])', m.group(0)) for k in ("correct", "rule_stated")}
        if all(d.values()):
            return {k: int(v.group(1)) for k, v in d.items()}
    return None

def parse_cond(raw):
    t = (raw or "").strip()
    m = list(re.finditer(r"KẾT?\s*LUẬN\s*[:：]\s*\**\s*([01])", t, flags=re.I))
    if m:
        return int(m[-1].group(1))
    if t[:1] in ("0", "1") and (len(t) == 1 or not t[1].isdigit()):
        return int(t[0])
    m = re.findall(r"(?m)^\s*([01])\s*$", t)
    return int(m[-1]) if m else None

def merge_judge(main, cond):
    if main is None or cond is None:
        return None
    return {"correct": main["correct"], "rule_stated": main["rule_stated"], "condition_kept": cond,
            "overgeneralized": int(main["rule_stated"] and not cond)}


# ---------------- generator ----------------
def hf_login():
    """Model gated trên Hugging Face: lấy token từ biến môi trường HF_TOKEN hoặc Kaggle Secrets (tên HF_TOKEN)."""
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = None
    if tok:
        os.environ["HF_TOKEN"] = tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
        print("[hf] đã nạp HF_TOKEN")
    else:
        print("[hf] không có HF_TOKEN — chỉ tải được model công khai")

class VLLMGenerator:
    def __init__(self, model, max_model_len=6144, gpu_util=0.90, tp=1):
        hf_login()
        from vllm import LLM
        print(f"[vllm] nạp {model} (tensor_parallel={tp})")
        self.llm = LLM(model, dtype="half", max_model_len=max_model_len, tensor_parallel_size=tp,
                       gpu_memory_utilization=gpu_util, enforce_eager=False, seed=42)

    def generate(self, batch_messages, max_tokens, temperature=0.0):
        from vllm import SamplingParams
        sp = SamplingParams(temperature=temperature, max_tokens=max_tokens, seed=42)
        outs = self.llm.chat(batch_messages, sp, use_tqdm=True)
        return [o.outputs[0].text for o in outs]

class MockGenerator:
    """--mock: chạy thử pipeline không cần GPU."""
    def generate(self, batch_messages, max_tokens, temperature=0.0):
        rng = random.Random(len(batch_messages))
        outs = []
        for m in batch_messages:
            if m[0]["content"].startswith("Bạn là người kiểm tra"):
                outs.append(f'TRÍCH: KHÔNG CÓ\nKẾT LUẬN: {rng.randint(0, 1)}')
            elif m[0]["content"].startswith("You are a strict"):
                outs.append("lý do...\n" + json.dumps({k: rng.randint(0, 1) for k in ("correct", "rule_stated")}))
            else:
                ctx = m[1]["content"].split("CÂU HỎI:")[0]
                outs.append(ctx[-300:])
        return outs


def run_batched(gen_factory, todo, build, max_tokens, out_path, key_fields, batch, deadline):
    if not todo:
        print(f"[run] không còn mục nào -> {out_path}"); return True
    gen = gen_factory()
    t0, n = time.time(), 0
    for i in range(0, len(todo), batch):
        if deadline and time.time() >= deadline:
            print(f"[checkpoint] chạm deadline sau {n}/{len(todo)} mục, dừng để lưu."); return False
        chunk = todo[i:i + batch]
        raws = gen.generate([build(it) for it in chunk], max_tokens=max_tokens)
        append_jsonl(out_path, [dict({k: it[k] for k in key_fields}, raw=r) for it, r in zip(chunk, raws)])
        n += len(chunk)
        print(f"[run] {n}/{len(todo)} — {(time.time() - t0) / n:.2f} s/mục", flush=True)
    return True


# ---------------- nạp dữ liệu ----------------
def load(a):
    cond_path = a.cond_file or find(a.root, "cond_questions_eval_1200.jsonl")
    assert cond_path and os.path.exists(cond_path), "Không thấy file câu hỏi cond (--cond_file)"
    qs = [q for q in read_jsonl(cond_path) if q.get("split", "eval") == "eval"]
    if a.limit:
        qs = qs[:a.limit]
    qmap = {q["qid"]: q for q in qs}
    print(f"[data] {len(qs)} câu cond từ {cond_path}")

    systems = list(a.systems) + (ABLATION_SYSTEMS if a.ablation else [])
    items = []
    seen = set()
    for s in systems:
        for B in a.budgets:
            cf = find(a.contexts_dir or a.root, f"contexts_{s}_B{B}.jsonl")
            if not cf:
                print(f"[!] thiếu contexts_{s}_B{B}.jsonl — bỏ qua"); continue
            got = 0
            for r in read_jsonl(cf):
                # contexts của s3 có thể chứa cùng qid 2 lần (bản cond + bản generic, nội dung như nhau) -> lấy 1
                if r["qid"] in qmap and f"{s}|B{B}|{r['qid']}" not in seen:
                    seen.add(f"{s}|B{B}|{r['qid']}")
                    items.append({"key": f"{s}|B{B}|{r['qid']}", "system": s, "budget": B,
                                  "qid": r["qid"], "units": r["units"]})
                    got += 1
            print(f"[data] {s} B{B}: {got} câu")
    if a.oracle:
        arts = {x["aid"]: x for x in read_jsonl(a.articles_file or find(a.root, "articles_all.jsonl") or find(a.root, "articles_1000.jsonl"))}
        for q in qs:
            txt = " ".join(arts[q["aid"]]["article_content"].split()[:a.oracle_words])
            items.append({"key": f"oracle|full|{q['qid']}", "system": "oracle", "budget": "full",
                          "qid": q["qid"], "units": [{"aid": q["aid"], "text": txt}]})
        print(f"[data] oracle (Điều gốc, ≤{a.oracle_words} từ): {len(qs)} câu")
    return qmap, items


# ---------------- các giai đoạn ----------------
def stage_reader(a, qmap, items, gen_factory, deadline):
    out = os.path.join(a.work, f"reader.shard{a.shard}.jsonl")
    done = {r["key"] for p in glob.glob(os.path.join(a.work, "reader.shard*.jsonl")) for r in read_jsonl(p)}
    todo = [it for it in items if shard_of(it["key"], a.num_shards) == a.shard and it["key"] not in done]
    print(f"[reader] shard {a.shard}/{a.num_shards}: còn {len(todo)} mục")
    return run_batched(gen_factory, todo, lambda it: reader_messages(qmap[it["qid"]]["question"], it["units"]),
                       a.reader_max_tokens, out, ["key", "system", "budget", "qid"], a.batch, deadline)

def judge_files(a):
    return sorted(glob.glob(os.path.join(a.work, f"judge_{a.judge_tag}.shard*.jsonl")))

def stage_judge(a, qmap, gen_factory, deadline):
    answers = {r["key"]: r for p in glob.glob(os.path.join(a.work, "reader.shard*.jsonl")) for r in read_jsonl(p)}
    out = os.path.join(a.work, f"judge_{a.judge_tag}.shard{a.shard}.jsonl")
    done = set()
    for p in judge_files(a):
        for r in read_jsonl(p):
            ok = parse_cond(r["raw"]) is not None if r["key"].endswith("#c") else parse_main(r["raw"]) is not None
            if ok:
                done.add(r["key"])
    todo = []
    for k, r in answers.items():
        if r["qid"] not in qmap or shard_of(k, a.num_shards) != a.shard:
            continue
        if k not in done:
            todo.append(dict(r, task="main"))
        if a.cond_source == "judge" and k + "#c" not in done:
            todo.append(dict(r, key=k + "#c", task="cond"))
    print(f"[judge] shard {a.shard}/{a.num_shards}: còn {len(todo)} lượt chấm (trên {len(answers)} câu trả lời)")
    build = lambda it: (judge_messages if it["task"] == "main" else cond_messages)(qmap[it["qid"]], it["raw"])
    return run_batched(gen_factory, todo, build, 256, out, ["key", "system", "budget", "qid"], a.batch, deadline)

def bootstrap_diff(x, y, iters=1000, seed=0):
    rng = random.Random(seed); n = len(x)
    if n == 0:
        return 0.0, 0.0, 0.0
    ds = []
    for _ in range(iters):
        idx = [rng.randrange(n) for _ in range(n)]
        ds.append(sum(x[i] - y[i] for i in idx) / n)
    ds.sort()
    return sum(p - q for p, q in zip(x, y)) / n, ds[int(.025 * iters)], ds[int(.975 * iters) - 1]

def stage_aggregate(a, qmap):
    answers = {r["key"]: r for p in glob.glob(os.path.join(a.work, "reader.shard*.jsonl")) for r in read_jsonl(p)}
    mains, conds = {}, {}
    for p in judge_files(a):
        for r in read_jsonl(p):
            if r["key"].endswith("#c"):
                v = parse_cond(r["raw"])
                if v is not None or r["key"][:-2] not in conds:
                    conds[r["key"][:-2]] = v
            else:
                v = parse_main(r["raw"])
                if v is not None or r["key"] not in mains:
                    mains[r["key"]] = v
    if a.cond_source == "phrase":   # mặc định: 'giữ điều kiện' đo bằng so khớp cụm từ (không dùng LLM)
        conds = {k: int(phrase_hit(qmap[r["qid"]]["condition"], r["raw"]) and not ABSTAIN_PAT.search(r["raw"]))
                 for k, r in answers.items() if r["qid"] in qmap}
    judged = {k: merge_judge(mains.get(k), conds.get(k)) for k in answers}

    per = defaultdict(dict)   # (system, budget) -> qid -> metrics
    n_fail = defaultdict(int)
    for k, r in answers.items():
        q = qmap.get(r["qid"])
        if q is None:
            continue
        ans = r["raw"]
        m = {"abstain": float(bool(ABSTAIN_PAT.search(ans))),
             "cond_cov_lex": float(coverage(q["condition"], [ans]) >= 0.6),
             "cond_phrase": float(phrase_hit(q["condition"], ans) and not ABSTAIN_PAT.search(ans))}
        j = judged.get(k)
        if j is None:
            n_fail[(r["system"], str(r["budget"]))] += 1
        else:
            m.update({"correct": float(j["correct"]), "condition_kept": float(j["condition_kept"]),
                      "cond_acc": float(j["correct"] and j["condition_kept"]),
                      "overgeneralized": float(j["overgeneralized"])})
        per[(r["system"], str(r["budget"]))][r["qid"]] = m

    cols = ["correct", "condition_kept", "cond_acc", "overgeneralized", "abstain", "cond_cov_lex", "cond_phrase"]
    # câu "cần điều kiện": điều kiện thật sự xuất hiện trong đáp án chuẩn (874/1200 ở bộ hiện tại)
    relevant = {k for k, q in qmap.items() if coverage(q["condition"], [q["answer"]]) >= 0.6}
    # kiểm tra độ tin cậy của judge: câu trả lời chứa gần nguyên văn điều kiện -> judge phải chấm kept=1
    # Kiểm tra độ tin cậy của judge (chỉ có ý nghĩa khi --cond_source judge)
    jud = [(k, r) for k, r in answers.items() if r["qid"] in qmap and judged.get(k) and not ABSTAIN_PAT.search(r["raw"])]
    pos = [k for k, r in jud if phrase_hit(qmap[r["qid"]]["condition"], r["raw"], frac=0.8)]      # điều kiện gần nguyên văn
    neg = [k for k, r in jud if coverage(qmap[r["qid"]]["condition"], [r["raw"]]) < 0.2]          # gần như không có điều kiện
    abst = [k for k, r in answers.items() if r["qid"] in qmap and judged.get(k) and ABSTAIN_PAT.search(r["raw"])]
    pos_rate = mean([float(judged[k]["condition_kept"]) for k in pos])
    neg_rate = mean([float(judged[k]["condition_kept"]) for k in neg])
    abst_correct = mean([float(judged[k]["correct"]) for k in abst])
    sanity_pass = (pos_rate == pos_rate and pos_rate >= 0.85) and (neg_rate == neg_rate and neg_rate <= 0.15)
    verb, sanity = pos, pos_rate
    res = {"reader_model": a.model, "judge_model": a.judge_model, "judge_tag": a.judge_tag,
           "n_questions": len(qmap), "n_condition_relevant": len(relevant),
           "judge_sanity": {"cond_source": a.cond_source,
                            "pos_n": len(pos), "pos_kept_rate": pos_rate,
                            "neg_n": len(neg), "neg_kept_rate": neg_rate,
                            "abstain_n": len(abst), "abstain_judged_correct": abst_correct,
                            "pass": bool(sanity_pass)},
           "table": {}, "table_relevant": {}, "significance": {}}
    for (s, B), d in per.items():
        row = {c: mean([m[c] for m in d.values() if c in m]) for c in cols}
        row.update({"n": len(d), "n_judge_fail": n_fail[(s, B)]})
        res["table"][f"{s}|B{B}"] = row
        dr = {k: m for k, m in d.items() if k in relevant}
        rr = {c: mean([m[c] for m in dr.values() if c in m]) for c in cols}
        rr.update({"n": len(dr), "n_judge_fail": 0})
        res["table_relevant"][f"{s}|B{B}"] = rr

    for metric in ["cond_acc", "correct", "overgeneralized", "cond_cov_lex", "cond_phrase"]:
        for (s, B), d in per.items():
            if s == "lp" or s == "oracle" or ("lp", B) not in per:
                continue
            L = per[("lp", B)]
            ks = sorted(k for k in set(L) & set(d) if metric in L[k] and metric in d[k])
            if ks:
                dlt, lo, hi = bootstrap_diff([L[k][metric] for k in ks], [d[k][metric] for k in ks], a.iters)
                res["significance"][f"{metric}@B{B}: lp - {s}"] = {"diff": dlt, "ci95": [lo, hi], "n": len(ks),
                                                                   "significant": not (lo <= 0 <= hi)}

    out_dir = os.path.join(a.work, a.results_dir); os.makedirs(out_dir, exist_ok=True)
    json.dump(res, open(os.path.join(out_dir, "s5_results.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)

    names = {"correct": "Accuracy", "condition_kept": "Cond. kept", "cond_acc": "Cond-Acc",
             "overgeneralized": "Over-general", "abstain": "Abstain", "cond_cov_lex": "CondCov (từ)", "cond_phrase": "CondCov (cụm)"}
    def render(tbl, title):
        out = [title, "", "| Hệ | Budget | " + " | ".join(names[c] for c in cols) + " | n | judge lỗi |", "|---" * (len(cols) + 4) + "|"]
        ks = sorted(tbl, key=lambda k: (str(k.split("|")[1]), ORDER.index(k.split("|")[0]) if k.split("|")[0] in ORDER else 99))
        for k in ks:
            s_, B_ = k.split("|"); r = tbl[k]
            out.append(f"| {s_} | {B_} | " + " | ".join(pct(r[c]) for c in cols) + f" | {r['n']} | {r['n_judge_fail']} |")
        return out + [""]
    if a.cond_source == "phrase":
        note = ("- 'Giữ điều kiện' (và Cond-Acc, Over-general) đo bằng so khớp cụm từ liền nhau giữa điều kiện và câu trả lời, "
                "không dùng LLM. Judge chỉ chấm 'Accuracy' (đúng/sai) và 'có nêu quy tắc'.")
    else:
        note = (f"- Kiểm tra judge ({'ĐẠT' if sanity_pass else 'CHƯA ĐẠT'}): "
                f"ca dương — {len(pos)} câu trả lời chứa ≥80% cụm từ của điều kiện, judge chấm 'có' = **{pct(pos_rate)}%** (cần ≥ 85%); "
                f"ca âm — {len(neg)} câu trả lời gần như không có điều kiện, judge chấm 'có' = **{pct(neg_rate)}%** (cần ≤ 15%); "
                f"câu trả lời 'không đủ thông tin' bị chấm đúng = {pct(abst_correct)}% (nên ≈ 0%).")
    L = [f"# Tầng 3 — Reader: {a.model} | Judge: {a.judge_model} ({a.judge_tag})", "", note, ""]
    L += render(res["table"], f"## Toàn bộ {len(qmap)} câu cond (%)")
    L += render(res["table_relevant"], f"## {len(relevant)} câu mà điều kiện xuất hiện trong đáp án chuẩn (%)")
    if res["significance"]:
        L += ["", f"## Paired Bootstrap ({a.iters} lần)", "", "| So sánh | Chênh lệch (điểm %) | CI 95% | n | Có ý nghĩa |", "|---|---|---|---|---|"]
        for name, d in res["significance"].items():
            L.append(f"| {name} | {100 * d['diff']:+.2f} | [{100 * d['ci95'][0]:+.2f}, {100 * d['ci95'][1]:+.2f}] | {d['n']} | {'✓' if d['significant'] else ''} |")
    md = "\n".join(L) + "\n"
    open(os.path.join(out_dir, "s5_tables.md"), "w", encoding="utf-8").write(md)
    print(md)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", choices=["reader", "judge", "aggregate"], required=True)
    ap.add_argument("--root", default="/kaggle/input")
    ap.add_argument("--work", default="/kaggle/working/work_s5")
    ap.add_argument("--model", default=DEFAULT_MODEL, help="model reader")
    ap.add_argument("--judge_model", default=None, help="model judge (mặc định = --model)")
    ap.add_argument("--cond_source", choices=["phrase", "judge"], default="phrase",
                    help="phrase: đo 'giữ điều kiện' bằng so khớp cụm từ (mặc định); judge: dùng LLM")
    ap.add_argument("--cond_file", default=None, help="v2: đường dẫn file câu hỏi cond")
    ap.add_argument("--contexts_dir", default=None, help="v2: thư mục contexts_* của s3_v2")
    ap.add_argument("--articles_file", default=None, help="v2: articles_v2.jsonl (cho hệ oracle)")
    ap.add_argument("--results_dir", default="results", help="thư mục con trong --work để ghi bảng kết quả")
    ap.add_argument("--judge_tag", default="v4", help="tên phiên bản judge (file judge_<tag>.shard*.jsonl)")
    ap.add_argument("--tp", type=int, default=1, help="tensor parallel; =2 nếu model không vừa 1 T4 (khi đó dùng --num_shards 1)")
    ap.add_argument("--max_model_len", type=int, default=6144)
    ap.add_argument("--systems", nargs="+", default=DEFAULT_SYSTEMS)
    ap.add_argument("--ablation", action="store_true", help="chạy thêm std_raw, lp_raw, lp_edge_only")
    ap.add_argument("--budgets", type=int, nargs="+", default=[100, 500])
    ap.add_argument("--oracle", action="store_true", default=True)
    ap.add_argument("--no_oracle", dest="oracle", action="store_false")
    ap.add_argument("--oracle_words", type=int, default=1500)
    ap.add_argument("--reader_max_tokens", type=int, default=384)
    ap.add_argument("--batch", type=int, default=256)
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--num_shards", type=int, default=1)
    ap.add_argument("--deadline_min", type=float, default=0, help="dừng an toàn sau N phút (0 = không giới hạn)")
    ap.add_argument("--limit", type=int, default=0, help="chỉ lấy N câu đầu để chạy thử")
    ap.add_argument("--iters", type=int, default=1000)
    ap.add_argument("--mock", action="store_true", help="chạy thử không cần GPU")
    a = ap.parse_args()
    os.makedirs(a.work, exist_ok=True)

    qmap, items = load(a)
    deadline = time.time() + a.deadline_min * 60 if a.deadline_min else None
    a.judge_model = a.judge_model or a.model
    use = a.judge_model if a.stage == "judge" else a.model
    gen_factory = (lambda: MockGenerator()) if a.mock else (lambda: VLLMGenerator(use, a.max_model_len, tp=a.tp))

    if a.stage == "reader":
        ok = stage_reader(a, qmap, items, gen_factory, deadline)
    elif a.stage == "judge":
        ok = stage_judge(a, qmap, gen_factory, deadline)
    else:
        stage_aggregate(a, qmap); ok = True
    print("[done]" if ok else "[chưa xong — chạy lại cùng lệnh để tiếp tục]")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/s6_evaluate.py
"""
s6_evaluate.py — Chấm Tầng 1 + Tầng 2 + Paired Bootstrap cho LP-Prop,
đọc lại ranks_*.jsonl / contexts_*.jsonl đã có từ s3 (KHÔNG encode lại).

Khác với phần s6 trong run_s3_s6.py:
  1. Luôn dùng ĐÚNG bộ 1.200 câu cond (cond_questions_eval_1200.jsonl của work_clean),
     không phụ thuộc thứ tự --inputs.
  2. Chấm "dedup-aware": các Điều có nội dung giống hệt nhau (133 nhóm) được coi là
     cùng một đáp án (Recall/MRR/CCR). Vẫn báo kèm Recall@5 strict (theo aid) để đối chiếu.
  3. Câu generic: bỏ câu thuộc Điều chỉ có tiêu đề (body < 5 từ) hoặc Điều không có
     passage nào, để so sánh công bằng giữa các mức hạt (áp dụng cho MỌI hệ).

Chạy trên Kaggle (add 3 dataset D1, S2, S3 làm input):
  python s6_evaluate.py --root /kaggle/input --out /kaggle/working/results
"""
import argparse, glob, hashlib, json, os, random, re, unicodedata
from collections import Counter, defaultdict

STRONG = {"TRỪ_TRƯỜNG_HỢP", "KHÔNG_ÁP_DỤNG", "VỚI_ĐIỀU_KIỆN", "NHƯNG_PHẢI", "GIỚI_HẠN"}
STOP = set("của và các là được có cho theo với trong này đó những một người phải thì".split())
ORDER = ["passage", "std_raw", "std", "lp_noptr", "lp_out", "lp_raw", "lp_edge_only", "lp"]
TIER1_SYS = ["std_raw", "std", "lp_raw", "lp_edge_only", "lp"]
BASES = ["passage", "std", "lp_noptr", "lp_out", "lp_raw", "lp_edge_only"]


# ---------------- tiện ích ----------------
def norm(s):
    return unicodedata.normalize("NFC", s or "")

def toks(s):
    return [w for w in re.findall(r"\w+", norm(s).lower()) if w not in STOP]

def coverage(span, texts):
    st = toks(span)
    if not st:
        return 0.0
    bag = set(w for t in texts for w in toks(t))
    return sum(1 for w in st if w in bag) / len(st)

def read_jsonl(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for l in f:
            l = l.strip()
            if l:
                try:
                    out.append(json.loads(l))
                except json.JSONDecodeError:
                    continue
    return out

def find(root, name, prefer=None):
    """Tìm file theo tên dưới root; nếu nhiều bản, ưu tiên đường dẫn chứa `prefer`."""
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
    if not hits:
        return None
    if prefer:
        pref = [h for h in hits if prefer in h]
        if pref:
            return pref[0]
    return hits[0]

mean = lambda xs: sum(xs) / len(xs) if xs else float("nan")
pct = lambda x: "–" if x != x else f"{100 * x:.1f}"


# ---------------- Tầng 1 (giữ nguyên công thức gốc) ----------------
def tier1(records, spans_by_aid, tau=0.6):
    by_aid = {r["aid"]: r for r in records}
    n = drop = uncond = keep_ptr = 0
    for aid, spans in spans_by_aid.items():
        r = by_aid.get(aid)
        if r is None:
            continue
        texts = {p["uid"]: p["text"] for p in r["props"]}
        for sp in spans:
            n += 1
            cond_any = any(coverage(sp["condition"], [t]) >= tau for t in texts.values())
            head_any = any(coverage(sp["head"], [t]) >= tau for t in texts.values())
            drop += not cond_any
            uncond += head_any and not cond_any
            self_c = any(coverage(sp["head"], [t]) >= tau and coverage(sp["condition"], [t]) >= tau
                         for t in texts.values())
            pair = any(coverage(sp["head"], [texts[e["head"]]]) >= tau and coverage(sp["condition"], [texts[e["tail"]]]) >= tau
                       for e in r.get("pointers", []) if e["head"] in texts and e["tail"] in texts)
            keep_ptr += self_c or pair
    n_props = sum(len(r["props"]) for r in records)
    words = [len(p["text"].split()) for r in records for p in r["props"]]
    return {
        "n_spans": n,
        "ConditionDropRate": drop / max(n, 1),
        "UnconditionalRuleRate": uncond / max(n, 1),
        "PointerRetention": keep_ptr / max(n, 1),
        "props_per_article": n_props / max(len(records), 1),
        "words_per_prop": sum(words) / max(len(words), 1),
        "pointers_per_article": sum(len(r.get("pointers", [])) for r in records) / max(len(records), 1),
    }


# ---------------- Tầng 2 (dedup-aware) ----------------
def recall_mrr(aids, gold_aid, grp, k=5):
    """grp: aid -> id nhóm nội dung. Hit nếu Điều trong top-k cùng nhóm với gold."""
    g = grp.get(gold_aid, gold_aid)
    for i, a in enumerate(aids[:k]):
        if grp.get(a, a) == g:
            return 1.0, 1.0 / (i + 1)
    return 0.0, 0.0

def recall_strict(aids, gold_aid, k=5):
    return float(gold_aid in aids[:k])

def ccr_item(q, ctx, grp, tau=0.6):
    g = grp.get(q["aid"], q["aid"])
    texts = [u["text"] for u in ctx if grp.get(u["aid"], u["aid"]) == g]
    h = coverage(q["head"], texts) >= tau
    c = coverage(q["condition"], texts) >= tau
    return {"head": float(h), "ccr": float(h and c), "orphan": float(h and not c)}

def bootstrap_diff(a, b, iters=1000, seed=0):
    rng = random.Random(seed); n = len(a)
    if n == 0:
        return 0.0, 0.0, 0.0
    ds = []
    for _ in range(iters):
        idx = [rng.randrange(n) for _ in range(n)]
        ds.append(sum(a[i] - b[i] for i in idx) / n)
    ds.sort()
    return sum(x - y for x, y in zip(a, b)) / n, ds[int(.025 * iters)], ds[int(.975 * iters) - 1]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="/kaggle/input", help="thư mục chứa 3 dataset D1, S2, S3")
    ap.add_argument("--out", default="/kaggle/working/results")
    ap.add_argument("--data", default=None, help="v2: thư mục data_v2 (articles_v2, passages, gold_spans, qa_generic, props_*)")
    ap.add_argument("--runs", default=None, help="v2: thư mục ranks_*/contexts_* của s3_v2")
    ap.add_argument("--cond", default=None, help="v2: file câu hỏi cond")
    ap.add_argument("--budgets", type=int, nargs="+", default=[100, 500])
    ap.add_argument("--k", type=int, default=5)
    ap.add_argument("--iters", type=int, default=1000)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    # ---- nạp dữ liệu ----
    D = a.data or a.root          # v2: --data trỏ thẳng tới data_v2 (không tìm lẫn sang dataset cũ)
    R = a.runs or a.root          # v2: --runs trỏ tới thư mục ranks/contexts của s3_v2
    arts_path = (find(D, "articles_v2.jsonl") or find(D, "articles_all.jsonl") or find(D, "articles_1000.jsonl"))
    cond_path = a.cond or find(a.root, "cond_questions_eval_1200.jsonl")
    assert cond_path, "Không thấy file câu hỏi cond (--cond)"
    paths = {
        "articles": arts_path,
        "cond": cond_path,
        "generic": find(D, "qa_generic.jsonl"),
        "gold": find(D, "gold_spans.jsonl"),
        "passages": find(D, "passages.jsonl"),
    }
    print("Đường dẫn dùng:", json.dumps(paths, ensure_ascii=False, indent=1))

    arts = {x["aid"]: x for x in read_jsonl(paths["articles"])}
    eval_aids = {k for k, v in arts.items() if v.get("split") == "eval"}

    # nhóm nội dung trùng
    h2a = defaultdict(list)
    for aid, x in arts.items():
        h2a[hashlib.md5(x["article_content"].strip().encode("utf-8")).hexdigest()].append(aid)
    grp = {aid: min(v) for v in h2a.values() for aid in v}
    n_dup_groups = sum(1 for v in h2a.values() if len(v) > 1)

    # Điều không dùng cho generic
    pas_aids = {p["aid"] for p in read_jsonl(paths["passages"])}
    heading_only = {k for k, v in arts.items() if len(v["article_content"].partition("\n")[2].split()) < 5}
    excl_generic = heading_only | (set(arts) - pas_aids)

    cond_list = [q for q in read_jsonl(paths["cond"]) if q["aid"] in eval_aids]
    gen_all = [q for q in read_jsonl(paths["generic"]) if q["aid"] in eval_aids]
    gen_list = [q for q in gen_all if q["aid"] not in excl_generic]
    cond_map = {q["qid"]: q for q in cond_list}
    gen_map = {q["qid"]: q for q in gen_list}
    n_cond_dup = sum(1 for q in cond_list if len(h2a[hashlib.md5(arts[q["aid"]]["article_content"].strip().encode("utf-8")).hexdigest()]) > 1)
    print(f"cond = {len(cond_list)} (trong đó {n_cond_dup} câu thuộc Điều có bản trùng) | "
          f"generic = {len(gen_list)} (đã bỏ {len(gen_all) - len(gen_list)} câu thuộc Điều rỗng/không passage)")

    spans = defaultdict(list)
    for s in read_jsonl(paths["gold"]):
        if s["aid"] in eval_aids and s["type"] in STRONG:
            spans[s["aid"]].append(s)

    res = {"split": "eval",
           "setup": {"n_cond": len(cond_list), "n_cond_on_dup_articles": n_cond_dup,
                     "n_generic": len(gen_list), "n_generic_dropped": len(gen_all) - len(gen_list),
                     "dup_groups": n_dup_groups, "k": a.k, "paths": paths},
           "tier1": {}, "tier2": {}, "significance": {}}

    # ---- Tầng 1 ----
    for m in TIER1_SYS:
        f = find(D, f"props_{m}.jsonl", prefer=None if a.data else "work_clean")
        if f:
            recs = [r for r in read_jsonl(f) if r["aid"] in eval_aids]
            res["tier1"][m] = tier1(recs, spans, 0.6)
            print(f"[Tier1] {m}: {res['tier1'][m]['ConditionDropRate']:.4f} drop")

    # ---- Tầng 2 ----
    vec = defaultdict(dict)
    for s in ORDER:
        rf = find(R, f"ranks_{s}.jsonl")
        if not rf:
            print(f"[!] thiếu ranks_{s}.jsonl — bỏ qua"); continue
        r_c, r_cs, r_g, r_gs, m_g = {}, [], [], [], []
        for r in read_jsonl(rf):
            q = cond_map.get(r["qid"]) or gen_map.get(r["qid"])
            if not q:
                continue
            rec, mrr = recall_mrr(r["aids"], q["aid"], grp, a.k)
            strict = recall_strict(r["aids"], q["aid"], a.k)
            if r["qid"] in cond_map:
                r_c[r["qid"]] = rec; r_cs.append(strict)
            else:
                r_g.append(rec); r_gs.append(strict); m_g.append(mrr)
        d = res["tier2"].setdefault(s, {})
        d.update({f"Recall@{a.k}_cond": mean(list(r_c.values())), f"Recall@{a.k}_cond_strict": mean(r_cs),
                  f"Recall@{a.k}_generic": mean(r_g), f"Recall@{a.k}_generic_strict": mean(r_gs),
                  f"MRR@{a.k}_generic": mean(m_g), "n_cond_ranked": len(r_c), "n_generic_ranked": len(r_g)})
        vec[f"Recall@{a.k}_cond"][s] = r_c

        for B in a.budgets:
            cf = find(R, f"contexts_{s}_B{B}.jsonl")
            if not cf:
                continue
            items = {r["qid"]: ccr_item(cond_map[r["qid"]], r["units"], grp, 0.6)
                     for r in read_jsonl(cf) if r["qid"] in cond_map}
            d[f"CCR@B{B}"] = mean([x["ccr"] for x in items.values()])
            d[f"Orphan@B{B}"] = mean([x["orphan"] for x in items.values()])
            d[f"HeadRecall@B{B}"] = mean([x["head"] for x in items.values()])
            d[f"n_ctx@B{B}"] = len(items)
            vec[f"CCR@B{B}"][s] = {k: x["ccr"] for k, x in items.items()}
            vec[f"Orphan@B{B}"][s] = {k: x["orphan"] for k, x in items.items()}
        print(f"[Tier2] {s}: n_cond={len(r_c)} R@{a.k}_cond={pct(d[f'Recall@{a.k}_cond'])}")

    # ---- Paired bootstrap: lp vs các hệ khác ----
    for metric, by_sys in vec.items():
        for base in BASES:
            if "lp" in by_sys and base in by_sys:
                A, Bv = by_sys["lp"], by_sys[base]
                ks = sorted(set(A) & set(Bv))
                if ks:
                    dlt, lo, hi = bootstrap_diff([A[k] for k in ks], [Bv[k] for k in ks], a.iters)
                    res["significance"][f"{metric}: lp - {base}"] = {
                        "diff": dlt, "ci95": [lo, hi], "n": len(ks), "significant": not (lo <= 0 <= hi)}

    json.dump(res, open(os.path.join(a.out, "results.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)

    # ---- bảng markdown ----
    k = a.k
    L = [f"# Kết quả LP-Prop — BGE-M3 (split = eval)", "",
         f"- Câu có điều kiện: **{len(cond_list)}** ({n_cond_dup} câu thuộc Điều có bản trùng nội dung; chấm dedup-aware)",
         f"- Câu generic: **{len(gen_list)}** (đã bỏ {len(gen_all) - len(gen_list)} câu thuộc Điều chỉ có tiêu đề / không có passage)",
         f"- Nhóm Điều trùng nội dung trong corpus: {n_dup_groups}", ""]
    if res["tier1"]:
        L += ["## Tầng 1 — Propositionizer", "",
              "| Hệ | Spans | Condition Drop (%) | Unconditional Rule (%) | Pointer Retention (%) | Props/Điều | Từ/prop | Pointers/Điều |",
              "|---|---|---|---|---|---|---|---|"]
        for m, d in res["tier1"].items():
            L.append(f"| {m} | {d['n_spans']} | {pct(d['ConditionDropRate'])} | {pct(d['UnconditionalRuleRate'])} | "
                     f"{pct(d['PointerRetention'])} | {d['props_per_article']:.2f} | {d['words_per_prop']:.2f} | {d['pointers_per_article']:.2f} |")
        L.append("")
    if res["tier2"]:
        cols = [f"Recall@{k}_cond", f"Recall@{k}_cond_strict", f"Recall@{k}_generic", f"MRR@{k}_generic"]
        for B in a.budgets:
            cols += [f"HeadRecall@B{B}", f"CCR@B{B}", f"Orphan@B{B}"]
        L += ["## Tầng 2 — Truy xuất BGE-M3 (%)", "", "| Hệ | " + " | ".join(cols) + " |", "|---" * (len(cols) + 1) + "|"]
        for s in [x for x in ORDER if x in res["tier2"]]:
            L.append(f"| {s} | " + " | ".join(pct(res["tier2"][s].get(c, float("nan"))) for c in cols) + " |")
        L.append("")
    if res["significance"]:
        L += [f"## Paired Bootstrap ({a.iters} lần)", "", "| So sánh | Chênh lệch (điểm %) | CI 95% | n | Có ý nghĩa |", "|---|---|---|---|---|"]
        for name, d in res["significance"].items():
            L.append(f"| {name} | {100 * d['diff']:+.2f} | [{100 * d['ci95'][0]:+.2f}, {100 * d['ci95'][1]:+.2f}] | {d['n']} | {'✓' if d['significant'] else ''} |")
    md = "\n".join(L) + "\n"
    open(os.path.join(a.out, "tables.md"), "w", encoding="utf-8").write(md)
    print("\n" + md)


if __name__ == "__main__":
    main()


In [ ]:
import os, subprocess, json, glob, time
def run(cmd, log=None, env=None):
    """Chạy lệnh trong /kaggle/working; nếu có log thì in dòng cuối của log."""
    full = f"cd /kaggle/working && {cmd}" + (f" > {log} 2>&1" if log else "")
    rc = subprocess.run(full, shell=True, env={**os.environ, **(env or {})}).returncode
    if log:
        lines = open(f"/kaggle/working/{log}", encoding="utf-8", errors="ignore").read().strip().splitlines()
        print(f"[{log}] " + (lines[-1] if lines else "(trống)"))
    return rc
def log_done(log):
    p = f"/kaggle/working/{log}"
    return os.path.exists(p) and open(p, encoding="utf-8", errors="ignore").read().rstrip().endswith("[done]")

# Tìm dữ liệu v2 (output của Notebook 1)
DATA = os.path.dirname(glob.glob("/kaggle/input/**/data_v2/articles_v2.jsonl", recursive=True)[0])
COND = glob.glob("/kaggle/input/**/questions_v2/cond_questions_v2_eval_1200.jsonl", recursive=True)[0]
S3 = "/kaggle/working/work_s3v2"
n_cond = sum(1 for _ in open(COND, encoding="utf-8"))
print("DATA =", DATA); print("COND =", COND, "|", n_cond, "câu")
assert n_cond == 1200, "File câu hỏi không đủ 1.200 câu — kiểm tra lại Notebook 1"
print("embedding cũ:", sorted(os.path.basename(p) for p in glob.glob("/kaggle/input/**/emb_*_bge-m3.npy", recursive=True)))

READER = "Qwen/Qwen2.5-7B-Instruct-AWQ"
JUDGE, JUDGE_TP = "Qwen/Qwen2.5-14B-Instruct-AWQ", 2
S5 = (f"python s5_reader_judge.py --cond_file {COND} --contexts_dir {S3} --articles_file {DATA}/articles_v2.jsonl "
      f"--root /kaggle/working/none --work /kaggle/working/work_s5 --model {READER}")

## s3: truy xuất

In [ ]:
run(f"python s3_retrieve_v2.py --data {DATA} --cond {COND} --old_emb_root /kaggle/input --out {S3}", log="s3.log")
print(open(f"{S3}/s3_meta.json", encoding="utf-8").read())

## s5: reader (2 GPU, khởi động lệch nhau 2 phút; phần nào lỗi thì chạy bù trên GPU 0)

In [ ]:
ps = []
for i in (0, 1):
    cmd = f"cd /kaggle/working && CUDA_VISIBLE_DEVICES={i} {S5} --stage reader --shard {i} --num_shards 2 --deadline_min 300 > reader{i}.log 2>&1"
    ps.append(subprocess.Popen(cmd, shell=True)); time.sleep(120)
for p in ps: p.wait()
for i in (0, 1): print(f"reader{i}:", "xong" if log_done(f"reader{i}.log") else "CHƯA XONG / LỖI")
for _ in range(3):   # chạy bù mọi câu còn thiếu trên GPU 0 (bỏ qua câu đã có)
    run(f"{S5} --stage reader --shard 0 --num_shards 1 --deadline_min 300", log="reader_fill.log", env={"CUDA_VISIBLE_DEVICES": "0"})
    if log_done("reader_fill.log"): break
keys = {json.loads(l)["key"] for f in glob.glob("/kaggle/working/work_s5/reader.shard*.jsonl") for l in open(f, encoding="utf-8") if l.strip()}
print("Số câu trả lời reader:", len(keys)); assert len(keys) == 13200

## s5: judge — kiểm tra trên 100 câu rồi chấm cả bộ

In [ ]:
SJ = f"{S5} --judge_model {JUDGE} --tp {JUDGE_TP}"
run(f"{SJ} --cond_source judge --stage judge --limit 100", log="judge_calib.log")
run(f"{SJ} --cond_source judge --stage aggregate --limit 100 --results_dir results_calib", log="agg_calib.log")
S = json.load(open("/kaggle/working/work_s5/results_calib/s5_results.json", encoding="utf-8"))["judge_sanity"]
print(json.dumps(S, ensure_ascii=False, indent=1))
COND_SRC = "judge" if S["pass"] else "phrase"
print(">>> JUDGE ĐẠT" if S["pass"] else ">>> JUDGE CHƯA ĐẠT -> 'giữ điều kiện' đo bằng cụm từ, judge chỉ chấm đúng/sai")
for _ in range(3):
    run(f"{SJ} --cond_source {COND_SRC} --stage judge --deadline_min 600", log="judge_full.log")
    if log_done("judge_full.log"): break

## Tổng hợp s5, chạy s6, nén kết quả

In [ ]:
run(f"{SJ} --cond_source {COND_SRC} --stage aggregate --results_dir results", log="agg_main.log")
run(f"{SJ} --cond_source phrase --stage aggregate --results_dir results_phrase", log="agg_phrase.log")
run(f"python s6_evaluate.py --data {DATA} --runs {S3} --cond {COND} --root /kaggle/working/none --out /kaggle/working/results_s6", log="s6.log")
!cd /kaggle/working && zip -qr lp_prop_v2_results.zip work_s5/results work_s5/results_phrase work_s5/results_calib work_s5/judge_v4.shard*.jsonl work_s5/reader.shard*.jsonl results_s6 work_s3v2/s3_meta.json *.log
!ls -la /kaggle/working/lp_prop_v2_results.zip

In [ ]:
from IPython.display import Markdown, display
for p in ["/kaggle/working/work_s5/results/s5_tables.md", "/kaggle/working/work_s5/results_phrase/s5_tables.md", "/kaggle/working/results_s6/tables.md"]:
    display(Markdown(f"---\n**{p}**\n\n" + open(p, encoding="utf-8").read()))

Tải **`lp_prop_v2_results.zip`** trong tab Output của version đã lưu và gửi lại để kiểm tra.